Lab 4: LLMs and Prompt Engineering for Decision Support

Student Name: [Amy Addo] Student ID: [74012028]

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os


# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")


# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1")
MODEL = "llama-3.1-8b-instant"                # or your provider's model name

print("Client ready.")


Client ready.


In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    print("Token usage:", response.usage)

    return response.choices[0].message.content


# Call it once with a simple question
answer = ask_llm("What is the capital of France?")
# TODO: Print response.usage as well — how many tokens did your call consume?
print(answer)

Token usage: CompletionUsage(completion_tokens=8, prompt_tokens=48, total_tokens=56, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.060850981, prompt_time=0.003127242, completion_time=0.009137968, total_time=0.01226521)
The capital of France is Paris.


1. What is the difference between the system and user roles? Give an example of something that belongs in each.
The system role is
 the first instruction to the language model, and which defines the task or
role for the LLM, and sets overall tone and context for the LLM while the user role is the prompt or task given by the human user.

Eg.
System Role:
Claude is able to explain difficult concepts or ideas clearly.
It can also illustrate its explanations with examples, thought
experiments, or metaphors.

User Role:
Explain photosythesis


2. What is a token, roughly? Why do API providers bill per token rather than per request?

A token is an atomic unit an AI model reads.

API providers bill per token rather than per request because of the computational operation cost. With a large prompt more computal power would be used than a short one.
If it was done per request then it won't be fair to the different users who may be prompting differently with varying tokens used per their output.

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
# A good test question: "Suggest a name for a savings product for market traders in Accra."

print("Temperature 0.0")
for i in range(5):
    answer = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
    print(answer)

print("\nTemperature 1.2")
for i in range(5):
    answer = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
    print(answer)


# TODO: Print all 10 answers, grouped by temperature.


Temperature 0.0
Token usage: CompletionUsage(completion_tokens=218, prompt_tokens=56, total_tokens=274, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.075566496, prompt_time=0.008715405, completion_time=0.511532759, total_time=0.520248164)
Considering the target market of market traders in Accra, I would suggest the following name for a savings product:

1. **Makola Savings**: "Makola" is a popular market in Accra, and using its name would help the product connect with the local market traders.
2. **TradeSafe**: This name emphasizes the safety and security of the savings product, which is essential for market traders who need to manage their finances effectively.
3. **MarketMoola**: "Moola" is a Ghanaian slang for money, and "Market" clearly indicates the product's target audience.
4. **AccraSavings**: This name is straightforward and emphasizes the product's connection to the city of Accra.
5. **PesaPesa**: "Pesa" is a Swahili word for money, and "PesaPesa" m

1. What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

2. When the temperature was 0.0 the LLM gave the same answers everytime and correctly gave factual suggestions for the question
In the case of when the temperature was 1.2, the answers kept varying everytime and it eventually strayed of topic. It gave very creative answers although some didn't answer the question.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


6 letters loaded.


In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = """
Summarize this loan application:

{letter_text}
"""

prompt = SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"])
v1_l002 = ask_llm(prompt)


prompt = SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"])
v1_l006 = ask_llm(prompt)




# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_PROMPT_V2 = """
Summarize this loan application:

{letter_text}

"""

system_prompt = """
Summarize the loan application in a factual and neutral manner.
Do not invent, assume, or infer information that is not stated in the letter.
Include only information supported by the application.
Keep the summary concise, using 3-4 sentences.

"""

prompt2 = SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"])
v2_l002 = ask_llm(prompt2,system_prompt,temperature=0)

prompt2 = SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"])
v2_l006 = ask_llm(prompt2,system_prompt,temperature=0)



# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

print("L002")
print("V1:", v1_l002)
print("V2:", v2_l002)

print("\nL006")
print("V1:", v1_l006)
print("V2:", v2_l006)



Token usage: CompletionUsage(completion_tokens=101, prompt_tokens=135, total_tokens=236, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.061253834, prompt_time=0.013059798, completion_time=0.165848982, total_time=0.17890878)
Token usage: CompletionUsage(completion_tokens=95, prompt_tokens=137, total_tokens=232, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.061335732, prompt_time=0.009094677, completion_time=0.126238302, total_time=0.135332979)
Token usage: CompletionUsage(completion_tokens=56, prompt_tokens=179, total_tokens=235, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.060014521, prompt_time=0.012410228, completion_time=0.21471593, total_time=0.227126158)
Token usage: CompletionUsage(completion_tokens=75, prompt_tokens=181, total_tokens=256, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.170184345, prompt_time=0.012707576, completion_time=0.109475918, total_time=0.122183494)
L

1. What concrete problems did V1's output have that V2 fixed? Quote examples.

V2 was straight to the point and did not add unnecessary detail for example:
In V1
Reason for urgency: Business has been slow, but expects a pickup after the festive season

V1 also presented information such as Trustworthiness for Kofi, even though this was only the applicant's own claim. V2 handled this more carefully by capturing it as:
Trustworthiness: Self-assessed by the applicant.

Lastly, V1 did not explicitly tell the model not to invent information. V2 added this constraint, which helped prevent unsupported assumptions, such as believing that an applicant has a good credit history or collateral simply because they seem trustworthy. It also helps prevent the assumption that businesses will succeed or that repayment is guaranteed, as shown in the report for the payment plan for Kofi.

2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?            

In a loan application, invented information can directly affect a financial decision. If an LLM makes up a good credit history, successful business experience, reliable income, collateral, or a realistic repayment ability, a loan officer could incorrectly judge the applicant as lower-risk.
The failure mode is called hallucination in LLM literature.


In [6]:
import json
import re
import pandas as pd

# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

EXTRACT_PROMPT = """Your goal is to extract information from a loan application letter into a JSON object. You MUST return ONLY the JSON object, and nothing else.  The JSON object MUST have the following keys and data types:

*   applicant_name (string)
*   amount_ghs (number)
*   purpose (string)
*   monthly_profit_ghs (number or null)
*   has_collateral_or_guarantor (boolean)
*   repayment_months (number or null)

If a field is not stated in the letter, use `null`. Do not guess.

Here is one example:

LETTER:
Dear Sir/Madam,
I am writing to apply for a loan of 5000 GHS to expand my tailoring business. I make about 800 GHS profit each month. I have a guarantor. I intend to repay the loan in 12 months.
Sincerely,
Ama Mensah

JSON:
{{ "applicant_name": "Ama Mensah", "amount_ghs": 5000, "purpose": "expand my tailoring business", "monthly_profit_ghs": 800, "has_collateral_or_guarantor": true, "repayment_months": 12 }}

Now, extract the information from the following LETTER. Remember to return ONLY the JSON object.

LETTER:
"""





# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text, temperature=0):
    full_prompt = f"{EXTRACT_PROMPT}{letter_text}"

    try:
        response_content = ask_llm(
            user_prompt=full_prompt,
            temperature=temperature
        )

        cleaned_json_string = re.sub(
            r"```json\s*|\s*```",
            "",
            response_content,
            flags=re.DOTALL
        ).strip()

        extracted_data = json.loads(cleaned_json_string)
        return extracted_data

    except json.JSONDecodeError as e:
        print(f"Warning: Failed to parse JSON. Error: {e}")
        return None

    except Exception as e:
        print(f"Warning: Extraction failed: {e}")
        return None

extracted_results = []

for key, letter in LETTERS.items():
    print(f"\nProcessing Letter {key}:")

    result = extract_fields(letter)

    if result:
        result["letter_id"] = key
        extracted_results.append(result)
    else:
        print(f"Could not extract data for Letter {key}.")

df_extracted = pd.DataFrame(extracted_results)

display(df_extracted)


Processing Letter L001:
Token usage: CompletionUsage(completion_tokens=73, prompt_tokens=435, total_tokens=508, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.060598987, prompt_time=0.036852533, completion_time=0.066016407, total_time=0.10286894)

Processing Letter L002:
Token usage: CompletionUsage(completion_tokens=72, prompt_tokens=395, total_tokens=467, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.107867213, prompt_time=0.204968185, completion_time=0.136936652, total_time=0.341904837)

Processing Letter L003:
Token usage: CompletionUsage(completion_tokens=77, prompt_tokens=449, total_tokens=526, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.061971485, prompt_time=0.040242564, completion_time=0.074359334, total_time=0.114601898)

Processing Letter L004:
Token usage: CompletionUsage(completion_tokens=73, prompt_tokens=415, total_tokens=488, completion_tokens_details=None, prompt_tokens_details=None, que

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,feed and 500 new layers for my poultry farm,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


1. Why must the few-shot example NOT come from the six letters you are processing?  
The model could learn the expected answer for that particular letter from the example rather than demonstrating that the extraction prompt works generally. This makes the evaluation less reliable


2. Why "use null, do not guess" — what did the model do without that instruction?
Without the instruction, The model may try to make a reasonable assumption.
The assumption could be false leading to hallucination

3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?
We don't want the model to give different answers everytime it is run. The answers must be consistent.
That's why temperature=0 is a good choice for structured extraction, classification, and other tasks where consistency and factuality matter.
For creative tasks, however, temperature 0 can be undesirable because you often want variation in responses.

In [7]:

# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

BRIEF_PROMPT = """You are an assistant to a microfinance loan officer.

Review the loan application letter and the extracted information provided below.
Your job is to prepare a factual, neutral brief to help a HUMAN loan officer
evaluate the application.

IMPORTANT RULES:
- Base all strengths and risks on evidence stated in the letter or extracted JSON.
- Do not invent facts or make assumptions about the applicant.
- Do not treat subjective claims by the applicant as established facts.
- Clearly distinguish missing information from negative information.
- Do NOT make a final lending decision.
- Do NOT say "approve" or "reject". Final decisions are always made by a human loan officer.

Provide exactly these four sections:

1. Strengths
- List concrete strengths supported by the application.

2. Risks / Red Flags
- List concrete risks or concerns supported by the application.

3. Missing Information
- List important information or documents that the loan officer should request.

4. Suggested Next Step
- Give one practical next step, such as requesting additional documents,
  inviting the applicant for an interview, conducting a site visit, or flagging
  the application for senior review.

LETTER:
{letter}

EXTRACTED_JSON:
{extracted_json}

BRIEF:
"""

letter_ids = list(LETTERS.keys())

extracted_data = {
    letter_ids[i]: extracted_results[i]
    for i in range(len(extracted_results))
}

# Generate briefs for ALL SIX letters
all_briefs = {}

for letter_id in letter_ids:
    letter_content = LETTERS[letter_id]
    extracted_info = extracted_data.get(letter_id)

    if extracted_info is None:
        print(f"Could not find extracted information for {letter_id}.")
        continue

    extracted_str = json.dumps(extracted_info, indent=2)

    brief_prompt = BRIEF_PROMPT.format(
        letter=letter_content,
        extracted_json=extracted_str
    )

    brief_output = ask_llm(brief_prompt, temperature=0)

    # Store the generated brief
    all_briefs[letter_id] = brief_output


# Print only L001, L002, and L006
for letter_id in ["L001", "L002", "L006"]:
    print(f"\nBrief for {letter_id}")
    print(all_briefs[letter_id])

Token usage: CompletionUsage(completion_tokens=457, prompt_tokens=492, total_tokens=949, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.169144953, prompt_time=0.037198496, completion_time=1.04001399, total_time=1.077212486)
Token usage: CompletionUsage(completion_tokens=304, prompt_tokens=451, total_tokens=755, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.068245483, prompt_time=0.027504017, completion_time=0.475102136, total_time=0.502606153)
Token usage: CompletionUsage(completion_tokens=368, prompt_tokens=510, total_tokens=878, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.06518986, prompt_time=0.065662619, completion_time=0.870175526, total_time=0.935838145)
Token usage: CompletionUsage(completion_tokens=318, prompt_tokens=472, total_tokens=790, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.169899991, prompt_time=0.028297008, completion_time=0.61661886, total_time=0.644915868)

1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each?
Yes, the system identified some of the  strengths and red flags for both letters correctly.
However in the letter L006 the model states Kofi's age as a strength and makes an assumption about his potential but that fact is unsupported as age does not translate to energy and potential.
Also L003 was not part of the displayed  So I had to print it separately to compare them.



2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.

This is because the LLM does not have access to certain sensitive information and since it can also make mistakes and assumptions like as seen above, It is better for a human officer to review using relevant context and fairness considerations.

In [8]:
fields = list(GOLD["L001"].keys())
letters_to_evaluate = ["L001", "L003", "L006"]

table_data = []

for field in fields:
    row = [field]
    correct_count = 0

    for letter_id in letters_to_evaluate:
        gold_value = GOLD[letter_id][field]
        gold_name = GOLD[letter_id]["applicant_name"]

        extracted_row = df_extracted[
            df_extracted["applicant_name"].str.lower() == gold_name.lower()
        ].iloc[0]

        extracted_value = extracted_row[field]

        if field == "applicant_name":
            correct = gold_value.lower() == str(extracted_value).lower()
        else:
            correct = gold_value == extracted_value

        row.append("Correct" if correct else "Wrong")

        if correct:
            correct_count += 1

    accuracy = correct_count / len(letters_to_evaluate) * 100
    row.append(f"{accuracy:.2f}%")

    table_data.append(row)

accuracy_df = pd.DataFrame(
    table_data,
    columns=["Field", "L001", "L003", "L006", "Accuracy"]
)

display(accuracy_df)

,Field,L001,L003,L006,Accuracy
0,applicant_name,Correct,Correct,Correct,100.00%
1,amount_ghs,Correct,Correct,Correct,100.00%
2,purpose,Wrong,Wrong,Wrong,0.00%
3,monthly_profit_ghs,Correct,Correct,Wrong,66.67%
4,has_collateral_or_guarantor,Correct,Correct,Correct,100.00%
5,repayment_months,Correct,Correct,Correct,100.00%


In [9]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
# temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced:
# (a) valid JSON
# (b) identical values across runs.

def analyze_extraction_runs(results, temperature):
    valid_json_count = 0
    unique_results = set()

    for result in results:
        if result is not None:
            valid_json_count += 1


            result_string = json.dumps(result, sort_keys=True)
            unique_results.add(result_string)

    print(f"\n--- Results for Temperature = {temperature} ---")
    print(f"Valid JSON outputs: {valid_json_count} out of {len(results)}")
    print(f"Unique result variants: {len(unique_results)}")
    print(f"All 5 runs identical: {len(unique_results) == 1}")


letter_l004 = LETTERS["L004"]


# Run 5 times at temperature = 0
print("Running extraction 5 times at temperature = 0...")

runs_temp_0 = []
for _ in range(5):
    result = extract_fields(letter_l004, temperature=0)
    runs_temp_0.append(result)

analyze_extraction_runs(runs_temp_0, temperature=0)


# Run 5 times at temperature = 1.0
print("\nRunning extraction 5 times at temperature = 1.0...")
runs_temp_1 = []
for _ in range(5):
    result = extract_fields(letter_l004, temperature=1.0)
    runs_temp_1.append(result)

analyze_extraction_runs(runs_temp_1, temperature=1.0)

Running extraction 5 times at temperature = 0...
Token usage: CompletionUsage(completion_tokens=73, prompt_tokens=415, total_tokens=488, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.06278838, prompt_time=0.173244598, completion_time=0.125708294, total_time=0.298952892)
Token usage: CompletionUsage(completion_tokens=73, prompt_tokens=415, total_tokens=488, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.062361858, prompt_time=0.060390601, completion_time=0.072959742, total_time=0.133350343)
Token usage: CompletionUsage(completion_tokens=73, prompt_tokens=415, total_tokens=488, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.061868618, prompt_time=0.029687733, completion_time=0.127823749, total_time=0.157511482)
Token usage: CompletionUsage(completion_tokens=73, prompt_tokens=415, total_tokens=488, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.071085533, prompt_time=0.034754627, compl

In [10]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.



print("\nAdversarial Test 1")

letter_l001 = LETTERS["L001"]

system_prompt_v2 = """
You are an assistant to a microfinance loan officer.
Provide factual, neutral summaries with no invented details.
If information is not provided, explicitly state that it is not provided.
"""

hallucination_prompt = f"""
Summarize this loan application:

{letter_l001}

Also answer this question:
What is Akosua Mensah's credit score?
"""

summarizer_output = ask_llm(
    hallucination_prompt,
    system_prompt=system_prompt_v2,
    temperature=0
)

print("Output:")
print(summarizer_output)
output_lower = summarizer_output.lower()
if (
    "not mentioned" in output_lower
    or "not provided" in output_lower
    or "not available" in output_lower
    or "not stated" in output_lower
    or "does not provide" in output_lower
):
    print("Test 1 Result: PASS")
else:
    print("Test 1 Result: FAIL")


print("\nAdversarial Test 2")

irrelevant_text = """
Today's weather forecast: Sunny with a high of 25 degrees Celsius.
The temperature will fall to 15 degrees tonight. No rain is expected.
"""

extractor_output = extract_fields(irrelevant_text)
print("Extractor Output:")
print(
    json.dumps(extractor_output, indent=2)
    if extractor_output is not None
    else extractor_output
)
if (
    extractor_output is not None
    and all(value is None for value in extractor_output.values())
):
    print("Test 2 Result: PASS")
else:
    print("Test 2 Result: FAIL")


Adversarial Test 1
Token usage: CompletionUsage(completion_tokens=120, prompt_tokens=220, total_tokens=340, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.063177826, prompt_time=0.015259476, completion_time=0.169112304, total_time=0.18437178)
Output:
Summary of the loan application:

Akosua Mensah, a 12-year vendor at Makola Market, is applying for a GHS 8,000 loan to purchase a deep freezer and expand her business into frozen foods. She has a monthly profit of GHS 900 and has saved GHS 2,500 through a susu scheme over the past two years. Akosua plans to repay the loan in 20 months with a monthly installment of GHS 450. Her sister, a teacher, has agreed to act as her guarantor.

Credit score: Not provided.
Test 1 Result: PASS

Adversarial Test 2
Token usage: CompletionUsage(completion_tokens=47, prompt_tokens=338, total_tokens=385, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.061110684, prompt_time=0.044893235, completion_time=0.05

1. Report your extraction accuracy. Which field was hardest for the model and why?   

The model achieved high overall field accuracy across the six fields and three evaluated letters.
It showed the purpose field as the hardest but this was because gold-standard had shorter summaries of the purpose, while the model extracted the purpose using wording from the letters.

2. What did the reliability experiment show about temperature and production systems?   

At temperature 0 all the 5 runs were identical
At temperature 1 all the 5 runs produced 4 unique results showing inconsistency.
Therefore, for a production system performing structured information extraction, temperature = 0 is better as the output are more repeatable



3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?  

The system did not hallucinate in Test 1. When asked for Akosua's credit score, which was not in the letter, it correctly responded Credit score: Not provided.  
With the Test 2, it halluciinated. he problem is that has_collateral_or_guarantor should also have been null, because the weather report does not state whether the applicant has collateral or a guarantor. Returning false incorrectly interprets missing information as a negative fact.

To reduce this hallucination the prompt should be more explicit in given out instructions for example, Intepret missing data as null and not false
